# EXECUÇÕES INICIAIS

## IMPORTAÇÕES DE BIBLIOTECAS

In [5]:
import os
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
from sklearn.model_selection import train_test_split
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import lightgbm as lgb
import shap
import joblib 

c:\Users\Lucas\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## CONFIGURAÇÕES INICIAIS
- Seta a pasta da onde estão os arquivos e o Setup.xlsx onde descreve o que significa cada arquivo
- Seta tempo de remoção dos dados para garantir estado do teste


In [3]:
# ---------- Paths ----------
PASTA_GERAL = "Dados/"
PASTA_DADOS = PASTA_GERAL + "Arquivos/"
SETUP_TESTES_XLSX = "Testes.xlsx"
NOME_ARQUIVO_LOG = "TEST_X.CSV"

# ---------- Pré-Processamento ----------
TEMPO_LIMPEZA_INICIAL = 5  # segundos
TIMESTAMP_PARA_SEGUNDOS = 1e3  # converter timestamp para segundos
NOME_VARIAVEL_TEMPO = "Timestamp"

# ---------- Estados ----------
niveis_estados = {
    0: "Balanceado",
    1: "Desbalanceado"
}

## HELPERS
Carrega as definições das funções

### FUNÇÃO DE ANÁLISE DAS VARIÁVEIS ORIGINAIS
- A função descritiva tem como objetivo realizar uma análise exploratória entre uma variável explicativa e a variável resposta binária. Para variáveis contínuas, os valores são discretizados em quantis, permitindo observar como a taxa média da variável resposta varia ao longo das faixas da variável analisada. O gráfico combina a média da variável resposta por categoria (eixo primário) com a frequência de amostras em cada faixa (eixo secundário), possibilitando avaliar simultaneamente tendência, separação entre classes e distribuição dos dados.
- A função relatorio_missing gera um resumo descritivo da base de dados quanto à presença de valores ausentes. Ela apresenta o número total de linhas e colunas do conjunto de dados, bem como a porcentagem e a frequência absoluta de valores faltantes em cada variável, permitindo identificar rapidamente problemas de qualidade dos dados antes da etapa de modelagem.

In [4]:
def descritiva(df_, var, vresp='survived', max_classes=5):
    """
    Gera um gráfico descritivo da taxa de sobreviventes por categoria da variável especificada.
    
    Parâmetros:
    df : DataFrame - Base de dados a ser analisada.
    var : str - Nome da variável categórica a ser analisada.
    """
    
    df = df_.copy()
    
    if df[var].nunique()>max_classes:
        df[var] = pd.qcut(df[var], max_classes, duplicates='drop')
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    sns.pointplot(data=df, y=vresp, x=var, ax=ax1)
    
    # Criar o segundo eixo y para a taxa de sobreviventes
    ax2 = ax1.twinx()
    sns.countplot(data=df, x=var, palette='viridis', alpha=0.5, ax=ax2)
    ax2.set_ylabel('Frequência', color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    
    ax1.set_zorder(2)
    ax1.patch.set_visible(False)  # Tornar o fundo do eixo 1 transparente
    
    # Exibir o gráfico
    plt.show()
    
def relatorio_missing(df):
    print(f'Número de linhas: {df.shape[0]} | Número de colunas: {df.shape[1]}')
    return pd.DataFrame({'Pct_missing': df.isna().mean().apply(lambda x: f"{x:.1%}"),
                          'Freq_missing': df.isna().sum().apply(lambda x: f"{x:,.0f}").replace(',','.')})

# CARREGAR ARQUIVOS

## CARREGAR TESTES.XLSX

In [ ]:
# ---------- 1) Ler metadados do Excel ----------
df_meta = pd.read_excel(os.path.join(PASTA_GERAL, "Testes.xlsx"))

# Normaliza nomes de colunas
df_meta = df_meta.rename(columns={
    "ID TEST": "ID_TEST",
})

# Garante tipos
df_meta["ID_TEST"] = df_meta["ID_TEST"].astype(int)

# Cria um dicionário: id -> (vel, desb)
meta_map = (
    df_meta
    .set_index("ID_TEST")[["Velocity", "Imbalance"]]
    .to_dict(orient="index")
)



## CARREGAR OS DADOS LOG_X.CSV

In [7]:
dados_original_separados = []  # Lista para armazenar os dados lidos
dados_original_juntos = []     # Lista para armazenar os dados combinados

# ---------- 1) Definindo local do arquivo ----------
padrao = re.compile(NOME_ARQUIVO_LOG.replace("X", r"(\d+)"), re.IGNORECASE)

# ---------- 2) Lendo arquivos CSV ----------
for nome_arquivo in os.listdir(PASTA_DADOS):
    correspondencia = padrao.match(nome_arquivo)
    if correspondencia:
        id_teste = int(correspondencia.group(1))
        caminho_arquivo = os.path.join(PASTA_DADOS, nome_arquivo)
        
        # Lê o arquivo CSV
        df_dados = pd.read_csv(caminho_arquivo)
        
        # Adiciona colunas de metadados
        if id_teste in meta_map:
            df_dados["Velocity"] = meta_map[id_teste]["Velocity"]
            df_dados["Imbalance"] = meta_map[id_teste]["Imbalance"]
            df_dados["ID_TEST"] = id_teste
            
            dados_original_separados.append(df_dados)
            
            print(f"Lido arquivo: {nome_arquivo} com ID_TEST {id_teste}")
        else:
            print(f"Metadados não encontrados para ID_TEST {id_teste}")
            
# ---------- 3) Junta tudo em um único DataFrame ----------
dados_original_juntos = pd.concat(dados_original_separados, ignore_index=True)

Lido arquivo: TEST_1.csv com ID_TEST 1
Lido arquivo: TEST_10.csv com ID_TEST 10
Lido arquivo: TEST_11.csv com ID_TEST 11
Lido arquivo: TEST_12.csv com ID_TEST 12
Lido arquivo: TEST_13.csv com ID_TEST 13
Lido arquivo: TEST_14.csv com ID_TEST 14
Lido arquivo: TEST_15.csv com ID_TEST 15
Lido arquivo: TEST_16.csv com ID_TEST 16
Lido arquivo: TEST_17.csv com ID_TEST 17
Lido arquivo: TEST_18.csv com ID_TEST 18
Lido arquivo: TEST_19.csv com ID_TEST 19
Lido arquivo: TEST_2.csv com ID_TEST 2
Lido arquivo: TEST_20.csv com ID_TEST 20
Lido arquivo: TEST_21.csv com ID_TEST 21
Lido arquivo: TEST_22.csv com ID_TEST 22
Lido arquivo: TEST_23.csv com ID_TEST 23
Lido arquivo: TEST_24.csv com ID_TEST 24
Lido arquivo: TEST_25.csv com ID_TEST 25
Lido arquivo: TEST_26.csv com ID_TEST 26
Lido arquivo: TEST_27.csv com ID_TEST 27
Lido arquivo: TEST_28.csv com ID_TEST 28
Lido arquivo: TEST_29.csv com ID_TEST 29
Lido arquivo: TEST_3.csv com ID_TEST 3
Lido arquivo: TEST_30.csv com ID_TEST 30
Lido arquivo: TEST_31.

## CHECAGEM DOS DADOS

In [8]:
print(dados_original_juntos.head())
print(dados_original_juntos.columns)

   Timestamp  Accel_X  Accel_Y  Accel_Z  Gyro_X  Gyro_Y  Gyro_Z  Mag_X  Mag_Y  \
0          0      -48       36      996       0       0       0    214   -366   
1         10      -46       25     1005       0       0       0    213   -372   
2         20      -73       70      979       0       0       0    210   -372   
3         30      -56       56      988       0       0       0    213   -361   
4         40      -54       26      996       0       0       0    205   -366   

   Mag_Z  Velocity  Imbalance  ID_TEST  
0   -942         0          0        1  
1   -951         0          0        1  
2   -952         0          0        1  
3   -939         0          0        1  
4   -957         0          0        1  
Index(['Timestamp', 'Accel_X', 'Accel_Y', 'Accel_Z', 'Gyro_X', 'Gyro_Y',
       'Gyro_Z', 'Mag_X', 'Mag_Y', 'Mag_Z', 'Velocity', 'Imbalance',
       'ID_TEST'],
      dtype='object')
